# Introducción y Contexto

**Descripción del Dataset:** El dataset seleccionado proviene de Kaggle y trata sobre la salud mental y el agotamiento laboral (Burnout) en empleados. Representa métricas de carga laboral, apoyo social y niveles de fatiga.

**Justificación:** Este análisis es relevante porque permite identificar patrones estadísticos que preceden al agotamiento crónico, utilizando PySpark para el manejo eficiente de grandes volúmenes de datos laborales.

**Preguntas de Investigación e Hipótesis:**
1. **¿Relación carga/fatiga?** * *Hipótesis:* A mayor asignación de recursos, mayor fatiga mental.
2. **¿Diferencias por género?** * *Hipótesis:* El Burn Rate varía significativamente entre géneros.
3. **¿Impacto del modo de trabajo?** * *Hipótesis:* El trabajo remoto influye en la percepción de apoyo social.

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

# 1. Iniciar sesión (se reutiliza)
spark = SparkSession.builder.appName("ParcialJhona").getOrCreate()

# 2. Cargar el dataset con el nombre real
# IMPORTANTE: Revisa que el nombre coincida letra por letra
path = "/home/jovyan/work/data/tech_mental_health_burnout.csv"
df = spark.read.csv(path, header=True, inferSchema=True)

# 3. Verificar que cargó bien
print(f"¡Listo! Cargamos {df.count()} filas.")
df.printSchema()
df.show(5)

# 4. Ver los nulos para el punto 4.2
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

¡Listo! Cargamos 150000 filas.
root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- experience_years: double (nullable = true)
 |-- company_size: string (nullable = true)
 |-- work_mode: string (nullable = true)
 |-- work_hours_per_week: double (nullable = true)
 |-- overtime_hours: double (nullable = true)
 |-- meetings_per_day: double (nullable = true)
 |-- deadlines_missed: integer (nullable = true)
 |-- job_satisfaction: double (nullable = true)
 |-- manager_support: double (nullable = true)
 |-- work_life_balance: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- physical_activity_days: integer (nullable = true)
 |-- screen_time_hours: double (nullable = true)
 |-- caffeine_intake: integer (nullable = true)
 |-- social_support_score: double (nullable = true)
 |-- has_therapy: integer (nullable = true)
 |-- stress_level: double (nullable = true)
 |-- anxiety_score: double (nullable = true)

**Descripción del cargue de los datos:** El dataset cargó completico con las 150,000 filas, ademas el conteo de nulos me dió cero en todas las columnas, así que no nos toca hacer limpieza rara ni borrar datos vitales. Las columnas ya vienen con los tipos de datos listos (enteros y decimales) para empezar con los analisis.

In [11]:
from pyspark.sql.functions import skewness, kurtosis

# Estadisticas generales de las variables que nos importan
df.select("burnout_score", "anxiety_score", "experience_years").summary("mean", "stddev", "min", "max").show()

# Calculo de Asimetria y Curtosis para el punto 4.3
df.select(
    skewness("burnout_score").alias("Asimetria_Burnout"),
    kurtosis("burnout_score").alias("Curtosis_Burnout")
).show()

+-------+------------------+------------------+-----------------+
|summary|     burnout_score|     anxiety_score| experience_years|
+-------+------------------+------------------+-----------------+
|   mean| 2.114121333333336| 4.523763333333334|5.055010000000028|
| stddev|1.1126135181052292|1.2015059695091777|2.878601560169965|
|    min|               1.0|               1.0|              0.0|
|    max|               8.1|               9.6|             18.5|
+-------+------------------+------------------+-----------------+

+------------------+------------------+
| Asimetria_Burnout|  Curtosis_Burnout|
+------------------+------------------+
|0.8907337519143389|0.1593606269081227|
+------------------+------------------+



In [12]:
from pyspark.sql.functions import skewness, kurtosis

# Calculamos estadísticas clave para la columna principal
df.select(
    skewness("burnout_score").alias("Asimetria_Burnout"),
    kurtosis("burnout_score").alias("Curtosis_Burnout"),
    skewness("anxiety_score").alias("Asimetria_Ansiedad")
).show()

+------------------+------------------+--------------------+
| Asimetria_Burnout|  Curtosis_Burnout|  Asimetria_Ansiedad|
+------------------+------------------+--------------------+
|0.8907337519143389|0.1593606269081227|0.005580381808385975|
+------------------+------------------+--------------------+



In [13]:
from pyspark.sql import functions as F

# 1. IQR y Outliers para Burnout Score
quantiles = df.approxQuantile("burnout_score", [0.25, 0.75], 0.0)
q1, q3 = quantiles[0], quantiles[1]
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_count = df.filter((F.col("burnout_score") < limite_inferior) | (F.col("burnout_score") > limite_superior)).count()

print(f"IQR: {iqr} | Limite Superior Outliers: {limite_superior}")
print(f"Número de Outliers detectados: {outliers_count}")

# 2. Análisis Categórico: Frecuencias y Moda para Job Role
total_filas = 150000
frecuencias = df.groupBy("job_role").count().withColumn("relativa", (F.col("count") / total_filas) * 100)
frecuencias.orderBy(F.desc("count")).show()

# 3. Tabla de contingencia: Género vs Nivel de Burnout
df.stat.crosstab("gender", "burnout_level").show()

IQR: 1.7999999999999998 | Limite Superior Outliers: 5.5
Número de Outliers detectados: 748
+------------------+-----+------------------+
|          job_role|count|          relativa|
+------------------+-----+------------------+
| Software Engineer|37623|25.081999999999997|
|    Data Scientist|22442|14.961333333333332|
|       ML Engineer|18153|            12.102|
| Backend Developer|18079|12.052666666666667|
|Frontend Developer|17967|            11.978|
|            DevOps|14897| 9.931333333333333|
|   Product Manager|11925|              7.95|
|       QA Engineer| 8914| 5.942666666666667|
+------------------+-----+------------------+

+--------------------+----+-----+--------+
|gender_burnout_level|High|  Low|Moderate|
+--------------------+----+-----+--------+
|              Female|  24|52858|    7399|
|          Non-binary|   3| 6576|     854|
|                Male|  38|72186|   10062|
+--------------------+----+-----+--------+



## Interpretación de los momentos estadísticos
Bueno aca podemos hablar en primer caso **Sobre la Salud Mental** la cual el promedio de burnout es 2.11, pero lo que me preocupa es que el máximo llega a 8.1, mejor dicho, aunque la mayoría del equipo está "relajad@", hay un grupo que ya está en el límite y eso es lo que sube el promedio. Igual pasa con la ansiedad, que tiene picos de 9.6., 
El valor positivo de la **Asimetría (0.89)** me dice que la distribución tiene una "cola" hacia la derecha. Básicamente, hay una minoría de empleados con puntajes de estrés altísimos que se salen del promedio general ademas la **Curtosis (0.15)** Como está tan cerca de cero, me indica que los datos están bien repartidos (mesocúrtica). No hay un solo valor donde todo el mundo esté amontonado, sino que la gente respondió de forma variada en toda la encuesta.

ademas la evaluación de la Experiencia no deja ver que tenemos gente con 0 años en la empresa hasta veteranos de 18 años. Eso hace que el analicis sea mucho más real porque cubre todo el rango de carrera.

Bueno los (Outliers) Calculando el IQR, encontré que cualquier persona con un burnout_score por encima de 5.5 ya se sale de lo normal, ademas logre Detectar que 748 empleados que están en ese nivel crítico, son pocos comparado con los 150k pero son los que tienen el riesgo mas alto de renunciar o enfermarse mañana mismo. La Moda indiscutible en esta empresa es el Software Engineer, que representa el 25.08% de toda la gente. Casi que 1 de cada 4 empleados es programador por ende la sobre carga en el trabajo de estas personas. 

Al cruzar Género con Nivel de Burnout, me doy cuenta que el nivel "High" (el más peligroso) lo lideran los hombres con 38 casos, seguidos por las mujeres con 24, aunque en porcentaje es bajo nos confirma que el estrés extremo está atacando a los 2 generos sin importar por ultimo la Diversidad de Roles Tenemos 8 categorías únicas de cargos, Desde el Product Manager hasta el QA Engineer, la carga está repartida pero los roles técnicos (DevOps, Backend, ML) son los que llenan el grueso de la operación.

In [14]:
# Primero registramos la vista con el DataFrame que ya cargaste
df.createOrReplaceTempView("burnout_table")

# 1. Consulta con filtro: Empleados con niveles de ansiedad muy altos
print("Corte de empleados con alta ansiedad:")
spark.sql("SELECT * FROM burnout_table WHERE anxiety_score > 8").show(5)

# 2. Consulta con Group By: Promedio de Burnout Score por Género
print("Promedio de Burnout por Género:")
spark.sql("SELECT gender, AVG(burnout_score) as promedio_burnout FROM burnout_table GROUP BY gender").show()

# 3. Consulta con Order By: Puestos (Job Roles) con más horas extra
print("Top 5 Roles con más horas extra:")
spark.sql("SELECT job_role, AVG(overtime_hours) as promedio_extras FROM burnout_table GROUP BY job_role ORDER BY promedio_extras DESC").show(5)

# 4. Consulta Compleja: ¿El modo de trabajo afecta la depresión y satisfacción?
print("Impacto del Work Mode en salud mental:")
spark.sql("""
    SELECT work_mode, AVG(depression_score) as prom_depresion, AVG(job_satisfaction) as prom_satisfaccion
    FROM burnout_table 
    GROUP BY work_mode 
    ORDER BY prom_depresion DESC
""").show()

Corte de empleados con alta ansiedad:
+---+------+-----------------+----------------+------------+---------+-------------------+--------------+----------------+----------------+----------------+---------------+-----------------+-----------+----------------------+-----------------+---------------+--------------------+-----------+------------+-------------+----------------+-------------+-------------+-----------------------+
|age|gender|         job_role|experience_years|company_size|work_mode|work_hours_per_week|overtime_hours|meetings_per_day|deadlines_missed|job_satisfaction|manager_support|work_life_balance|sleep_hours|physical_activity_days|screen_time_hours|caffeine_intake|social_support_score|has_therapy|stress_level|anxiety_score|depression_score|burnout_score|burnout_level|seeks_professional_help|
+---+------+-----------------+----------------+------------+---------+-------------------+--------------+----------------+----------------+----------------+---------------+------------

## Análisis de resultados con Spark SQL

Después de procesar las 150,000 filas con SQL, saqué estas conclusiones que me parecen las más importantes del ejercicio:

Al filtrar por niveles de ansiedad mayores a 8, me di cuenta que hay mucha gente en cargos técnicos como **Data Scientists y Software Engineers** que están al límite, lo mas curioso es que algunos tienen buena satisfacción laboral pero el estrés y la ansiedad los tienen volando, lo que confirma que el burnout es un problema silencioso.

En el Burnout por Género esta parte me tumbó la hipótesis que tenía, ya que los promedios son casi calcados 2.114 para mujeres, 2.114 para hombres y 2.110 para no binarios, con esto queda claro que en esta empresa el agotamiento es un tema general de la cultura de trabajo y no tiene nada que ver con el género de la persona, por lo cual tanto como hombre como mujeres sienten el mismos agotamiento o extres.  

Los más **"esclavizados"**  son los Backend Developers con 6.15 horas extra promedio. Les siguen muy de cerca los Data Scientists y los Software Engineers, básicamente entre más técnico el rol más horas le están metiendo por fuera de la jornada. ya para culminar el impacto del modo de trabajo esta fue la consulta más compleja y me soltó un dato vital el cual dice que los que trabajan Onsite (presencial) son los que tienen el promedio de depresión más alto (3.10) y la satisfacción más bajita (5.48). En cambio, los de modo Hybrid tienen la mejor satisfacción (5.51) esto me confirma lo que siempre e pensado y es que la flexibilidad de no ir todos los días a la oficina ayuda bastante al ánimo del equipo, ya que se puede compartir mayor tiempo con amigos, familiares he incluso evitar el estres y desespero del trasporte publico.

In [15]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

# Seleccionamos las variables numéricas clave
numeric_cols = ["age", "experience_years", "work_hours_per_week", "anxiety_score", "burnout_score"]

# Convertimos a vector para que Spark lo entienda
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vector = assembler.transform(df).select("features")

# Calculamos la matriz de Pearson
matrix = Correlation.corr(df_vector, "features").collect()[0][0]
correlations = matrix.toArray().tolist()

print("Matriz de Correlación (Pearson):")
for i, col_name in enumerate(numeric_cols):
    print(f"{col_name}: {correlations[i]}")

Matriz de Correlación (Pearson):
age: [1.0, -0.0013170127690478324, -0.002374132971661032, 0.00030266129269796056, -0.0018000967053876738]
experience_years: [-0.0013170127690478324, 1.0, 0.0036968598827991166, 3.6096824233365426e-05, -0.0006913271772488732]
work_hours_per_week: [-0.002374132971661032, 0.0036968598827991166, 1.0, 0.16256050041748996, 0.43990109159000096]
anxiety_score: [0.00030266129269796056, 3.6096824233365426e-05, 0.16256050041748996, 1.0, 0.5488478859460704]
burnout_score: [-0.0018000967053876738, -0.0006913271772488732, 0.43990109159000096, 0.5488478859460704, 1.0]


## Interpretación de la Matriz de Correlación. 
¿Qué variables están conectadas realmente?
Al revisar la matriz de Pearson, identifiqué las 3 relaciones más fuertes que explican por qué la gente se está quemando en la empresa:
**Ansiedad vs. Burnout** (0.54 - Positiva Fuerte) Esta es la conexión más influyente del estudio donde nos dice que a medida que sube el nivel de ansiedad, el burnout se dispara casi que al mismo tiempo. No es solo cansancio, es un tema mental serio. asi mismo las **Horas de Trabajo vs. Burnout** (0.43 - Tiene una correlación positiva Moderada) y aquí se confirma que trabajar de más sí pasa factura. Entre más suben las horas a la semana (work_hours_per_week), más alto es el puntaje de agotamiento, lo cual si es algo como muy simple de entender. 

Uno de los detaññes importantes es que Aunque veamos que la ansiedad y el burnout suben de la mano, Correlación NO implica Causalidad. aunque se que se muevan juntas no significa que la ansiedad sea la única culpable del burnout. Puede que haya una tercera variable (como la falta de sueño o problemas familiares) que esté afectando a ambas al tiempo y nosotros solo estemos viendo el resultado en los números.

# Conclusiones: #

1.En Respuesta a las preguntas de investigación puedo concuir lo siguiente: 
**¿El modo de trabajo influye?** RTA: Totalmente los datos me mostraron que la gente que trabaja Onsite (presencial) tiene los niveles de depresión más altos (3.10) comparado con los remotos o híbridos. Parece que la falta de flexibilidad pega duro en el ánimo.

**¿Hay roles más quemados que otros?** RTA: Sí los roles técnicos son los más sacrificados, Los Software Engineers y Backend Developers no solo son la mayoría en la empresa, sino que lideran las horas extra con promedios de 6.15 horas, lo que vitalmente sube su riesgo de burnout.

**¿La ansiedad explica el agotamiento?** RTA: En este caso la matriz de correlación me dio un 0.54, lo que es una relación vital. A medida que sube la ansiedad, el burnout se dispara. No es solo cansancio de sueño, es un tema de salud mental.

## Validación de Hipótesis # 
Hipótesis de Género: Yo pensaba que algún género estaría más afectado, pero los promedios fueron casi calcados (2.11 para todos), El burnout en tecnología no distingue si eres hombre, mujer o no binario; los afecta a todos por igual, por ende rechazamos esa hipotesis. 

Hipótesis de Carga Laboral: La correlación de 0.43 entre horas de trabajo y burnout confirma que entre más tiempo pasamos pegados al código más nos desgastamos, la evidencia está en los 748 outliers que detectamos con puntajes críticos de estrés por ende esta hipotesis es aceptada. 

## Limitaciones del Analisis
Sesgo de Sector: Los datos son puramente de empresas "Tech", así que no podemos decir que esto mismo pase en medicina o construcción. ademas la representatividad Aunque 150k filas es un montón, no sabemos se puede determinar si son son datos de Colombia o de todo el mundo, lo que puede cambiar la percepción del trabajo presencial. y ya por ultimo el dataset venía con 0 nulos, lo cual es fue un ahorro para el proyecto, pero todos tenemos claro que en muchos de los casos la data suele estar más sucia y eso podría darnos resultados diferentes.

Pues una de mis puntos a mejorar que se podria adactar para tener un buen proyecto seria implementar Modelos Predictivos que usen un modelo de clasificación que nos avise qué empleado tiene riesgo de burnout antes de que se convierta en un "outlier" y utilizar series de Tiempo para analizar cómo cambian estos niveles de estrés en fechas de cierre de sprint o entregas finales de semestre, para ver si hay picos estacionales.